In [ ]:
import json
import math
import random
import re
import time
from collections import Counter, defaultdict
from collections.abc import Callable, Collection, Hashable, Iterable, Mapping
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Any, Literal, cast

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from openai import OpenAI
from openai.types.chat import ChatCompletion, ChatCompletionMessage, ChatCompletionMessageParam
from openai.types.chat.chat_completion import Choice
from pydantic import BaseModel, ConfigDict, Field, ValidationError
from transformers import AutoConfig, AutoTokenizer

In [ ]:
# client, MODEL и NO_THINK задаются в разделе о сервере
# поля vLLM, которых нет в типах OpenAI SDK, помечены type: ignore[attr-defined]
pd.set_option("display.max_colwidth", 250)

Message = Mapping[str, Any]


def chat(
    messages: Iterable[Message],
    think: bool = False,
    extra_body: dict[str, Any] | None = None,
    **params: Any,
) -> ChatCompletion:
    """Ответ модели целиком, вместе с usage; по умолчанию жадный выбор без рассуждения."""
    params = {
        "temperature": 0,
        "max_tokens": 1000,
        **params,
    }
    base = {} if think else NO_THINK
    extra_body = base | (extra_body or {})

    return client.chat.completions.create(
        model=MODEL,
        messages=cast(Iterable[ChatCompletionMessageParam], messages),
        extra_body=extra_body,
        **params,
    )


def parallel[Item, Result](fn: Callable[[Item], Result], items: Collection[Item]) -> list[Result]:
    """fn для каждого элемента; запросы идут параллельно."""
    with ThreadPoolExecutor(max(1, len(items))) as pool:
        results = pool.map(fn, items)

        return list(results)


def chat_parallel(dialogs: Collection[Iterable[Message]], **params: Any) -> list[str]:
    """Ответ модели на каждый список messages."""
    responses = parallel(lambda messages: chat(messages, **params), dialogs)

    return [response.choices[0].message.content or "" for response in responses]


def rendered(messages: Iterable[Message], extra_body: dict[str, Any] | None = None, **params: Any) -> str:
    """Текст для модели: chat template применяет сервер."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=cast(Iterable[ChatCompletionMessageParam], messages),
        max_tokens=1,
        **params,
        extra_body={"return_prompt_text": True} | (extra_body or {}),
    )

    return response.prompt_text


def show(texts: Mapping[str, object]) -> None:
    for name, text in texts.items():
        print(f"=== {name}\n{text}\n")

## Модель

[Qwen3.5-4B](https://huggingface.co/Qwen/Qwen3.5-4B): 4,2 млрд параметров в языковой части, контекст до 262 144 токенов.

Модель мультимодальная (vision-language): ещё 0,3 млрд параметров приходится на визуальный энкодер.

Для примеров используем квантованную версию [RedHatAI/Qwen3.5-4B-quantized.w8a8](https://huggingface.co/RedHatAI/Qwen3.5-4B-quantized.w8a8).

```bash
uv run hf download RedHatAI/Qwen3.5-4B-quantized.w8a8
```

Свободной оперативной памяти желательно иметь около 16Gb.

## Запуск сервера

```bash
export VLLM_CPU_KVCACHE_SPACE=4
export VLLM_USE_V2_MODEL_RUNNER=0
uv run vllm serve RedHatAI/Qwen3.5-4B-quantized.w8a8 \
    --language-model-only \
    --max-model-len 8192 \
    --reasoning-parser qwen3 \
    --enable-auto-tool-choice \
    --tool-call-parser qwen3_coder \
    --generation-config vllm \
    --max-logprobs -1 \
    --no-enable-chunked-prefill \
    --max-num-batched-tokens 8192 \
    --no-enable-prefix-caching \
    --structured-outputs-config '{"backend": "xgrammar", "disable_any_whitespace": true}' \
    --port 8000
```

| Параметр | Зачем |
|---|---|
| `VLLM_CPU_KVCACHE_SPACE=4` | сколько ГБ оперативной памяти выделить под KV-кэш |
| `VLLM_USE_V2_MODEL_RUNNER=0` | исполнять модель прежним Model Runner, а не новым V2 |
| `--language-model-only` | не загружать визуальный энкодер |
| `--max-model-len 8192` | предел промпта и ответа вместе; KV-кэш должен вместить хотя бы один запрос такой длины |
| `--reasoning-parser qwen3` | сервер отделяет рассуждение `<think>…</think>` от ответа и возвращает его в поле `reasoning` |
| `--enable-auto-tool-choice` | модель сама решает, вызвать ли инструмент из запроса |
| `--tool-call-parser qwen3_coder` | сервер разбирает вызовы инструментов в формате Qwen3.5 и возвращает их в поле `tool_calls` |
| `--generation-config vllm` | не подставлять параметры сэмплирования из `generation_config.json` модели: по умолчанию действуют нейтральные значения vLLM |
| `--max-logprobs -1` | разрешить запрашивать логарифмы вероятностей для всего словаря |
| `--no-enable-chunked-prefill` | обрабатывать промпт целиком за один шаг, не разбивая на части |
| `--max-num-batched-tokens 8192` | сколько токенов сервер обрабатывает за один шаг; не меньше `--max-model-len`, чтобы промпт любой длины помещался в шаг целиком |
| `--no-enable-prefix-caching` | не переиспользовать KV-кэш общих начал промптов между запросами |
| `--structured-outputs-config '{"backend": "xgrammar", "disable_any_whitespace": true}'` | генерацию по JSON Schema ограничивает библиотека xgrammar; JSON -- без пробелов и переводов строк между полями |
| `--port 8000` | порт HTTP-сервера |


В логе запуска строка `CPU KV cache size` сообщает, сколько токенов поместилось в KV-кэш, а `Maximum concurrency` -- сколько запросов полной длины сервер может держать одновременно. Пока идут запросы, сервер каждые 10 секунд печатает строку `Engine 000: Avg prompt throughput ...`: скорость обработки промптов и генерации в токенах в секунду, число запросов в работе и в очереди, заполнение KV-кэша.

## Проверяем сервер

API сервера -- обычный HTTP с JSON, поэтому обращаться к нему можно через curl.

In [ ]:
%%bash
curl -sS -i http://localhost:8000/health

`/health` отвечает кодом 200, когда сервер готов.

`/v1/models` возвращает имя модели и предел контекста `max_model_len`.

In [ ]:
%%bash
curl -sS http://localhost:8000/v1/models | jq

Из Python удобнее работать через официальный клиент OpenAI: он умеет обращаться к любому совместимому серверу, достаточно указать `base_url`.

In [ ]:
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
    timeout=3600,
)

models = client.models.list()
MODEL = models.data[0].id
print(MODEL)

## Первый запрос: Chat Completions

Запрос состоит из списка сообщений и параметров генерации.

In [ ]:
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

messages: list[ChatCompletionMessageParam] = [
    {
        "role": "user",
        "content": "Объясни в двух предложениях, что такое языковая модель.",
    },
]
response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    max_tokens=200,
    temperature=0,
    extra_body=NO_THINK,
)

print(response.choices[0].message.content)

Ответ -- тоже JSON. Кроме текста в `choices[0].message.content`, в нём есть причина остановки `finish_reason` и расход токенов `usage`.

In [ ]:
print(response.model_dump_json(indent=2, exclude_none=True))

Клиент OpenAI лишь собирает такой JSON и отправляет его POST-запросом на `/v1/chat/completions`. Тот же запрос через curl: `-d` передаёт тело запроса, и curl отправляет его методом POST, а заголовок `Content-Type` сообщает серверу, что тело -- JSON.

In [ ]:
%%bash
curl -sS http://localhost:8000/v1/chat/completions -H "Content-Type: application/json" -d '{
  "model": "RedHatAI/Qwen3.5-4B-quantized.w8a8",
  "messages": [
    {
      "role": "user",
      "content": "Назови три цвета радуги."
    }
  ],
  "max_tokens": 50,
  "temperature": 0,
  "chat_template_kwargs": {"enable_thinking": false}
}' | jq

## Completions: продолжение текста

Кроме чата у сервера есть старый эндпоинт `/v1/completions`. Он получает строку `prompt` и продолжает её как есть, без шаблона сообщений. Такой режим удобен, чтобы смотреть на голую языковую модель: в ноутбуке 01 с его помощью будем изучать вероятности следующего токена.

In [ ]:
completion = client.completions.create(
    model=MODEL,
    prompt="Москва -- столица",
    max_tokens=12,
    temperature=0,
)

print(repr(completion.choices[0].text))

## Метрики сервера

На `/metrics` сервер публикует счётчики в формате Prometheus: число запросов в работе и в очереди, заполнение KV-кэша, всего обработано токенов промптов и ответов.

In [ ]:
%%bash
curl -sS http://localhost:8000/metrics | grep -E '^vllm:(num_requests_(running|waiting)|kv_cache_usage_perc|(prompt|generation)_tokens_total)\{'

In [ ]:
served = models.data[0]
tokenizer = AutoTokenizer.from_pretrained(served.root)  # type: ignore[attr-defined]
config = AutoConfig.from_pretrained(served.root)  # type: ignore[attr-defined]
VOCAB = config.get_text_config().vocab_size


def piece(token_id: int | np.integer) -> str:
    """Текст токена; пробел показан как ▁, перевод строки как \\n."""
    text = cast(str, tokenizer.decode([int(token_id)]))

    return text.replace(" ", "▁").replace("\n", "\\n")

## Токены

Для доступа к токенизатору vLLM предоставляет эндпоинт `/tokenize`.

In [ ]:
%%bash
curl -sS http://localhost:8000/tokenize -H "Content-Type: application/json" -d '{
  "model": "RedHatAI/Qwen3.5-4B-quantized.w8a8",
  "prompt": "Кот спит на"
}' | jq -c

Токеном может быть часть слова или слово вместе с пробелом. Чем слово популярнее, тем более вероятно, что оно будет представлено одним токеном.

> ... the most token efficient language humanity ever invented

In [ ]:
ids = tokenizer("Кот спит на")["input_ids"]
print(ids)

pieces = [piece(i) for i in ids]
print(" | ".join(pieces))

Номера токенов можно передать модели вместо текста: `prompt` в `/v1/completions` принимает и строку, и список номеров.

In [ ]:
%%bash
curl -sS http://localhost:8000/v1/completions -H "Content-Type: application/json" -d '{
  "model": "RedHatAI/Qwen3.5-4B-quantized.w8a8",
  "prompt": "Кот спит на",
  "max_tokens": 8,
  "temperature": 0
}' | jq .choices[0].text

curl -sS http://localhost:8000/v1/completions -H "Content-Type: application/json" -d '{
  "model": "RedHatAI/Qwen3.5-4B-quantized.w8a8",
  "prompt": [25497, 12666, 80730, 9080, 12696],
  "max_tokens": 8,
  "temperature": 0
}' | jq .choices[0].text

## Распределение следующего токена

Для переданного контекста модель вычисляет логиты следующего токена -- по одному числу на каждый токен словаря, -- а softmax затем превращает их в вероятности. Сервер возвращает логарифмы вероятностей (logprobs) для заданного числа самых вероятных токенов. Запросим весь словарь.

In [ ]:
def next_token_logprobs(prompt: str, k: int = VOCAB) -> np.ndarray:
    """Logprobs следующего токена после переданного промпта."""
    response = client.completions.create(
        model=MODEL,
        prompt=prompt,
        max_tokens=1,
        temperature=0,
        logprobs=k,
        extra_body={"return_tokens_as_token_ids": True},
    )
    choice = response.choices[0]
    assert choice.logprobs and choice.logprobs.top_logprobs
    top = choice.logprobs.top_logprobs[0]

    logprobs = np.full(VOCAB, -np.inf)
    for token, logprob in top.items():
        token_id = int(token.removeprefix("token_id:"))
        logprobs[token_id] = logprob

    return logprobs


def plot_distributions(distributions: dict[str, np.ndarray], title: str = "", k: int = 8) -> None:
    """Вероятности k самых вероятных токенов."""
    first = next(iter(distributions.values()))
    ids = np.argsort(-first)[:k]

    columns = {name: [*p[ids], 1 - p[ids].sum()] for name, p in distributions.items()}
    index = [piece(i) for i in ids] + ["остальные"]
    table = pd.DataFrame(columns, index=index)

    table.plot.bar(
        figsize=(9, 3),
        rot=60,
        title=title,
        ylabel="вероятность",
    )
    plt.show()


probs = np.exp(next_token_logprobs("Кот спит на"))
plot_distributions(
    {"модель": probs},
    title=f'После "Кот спит на": Размер словаря {VOCAB}, сумма для n > k {probs.sum():.3f}',
)

## Жадный выбор и beam search

Выбранный токен добавляется к контексту, и шаг повторяется. Вероятность последовательности -- произведение вероятностей её токенов, то есть сумма их logprobs (помним про знаменитый прием из статистики).

Жадный выбор на каждом шаге берёт самый вероятный токен.

Beam search хранит несколько самых вероятных последовательностей, в vLLM его можно включить параметром `use_beam_search`.

Продолжим "Кот спит на" на четыре токена обоими способами и посчитаем вероятности продолжений. Logprob каждого токена сервер возвращает для уже написанного текста: параметр `prompt_logprobs`.

In [ ]:
def generate_ids(prompt: list[int], max_tokens: int, n: int = 1, **vllm_params: Any) -> list[list[int]]:
    """Номера токенов n ответов при temperature 0."""
    response = client.completions.create(
        model=MODEL,
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=0,
        n=n,
        extra_body={
            "return_token_ids": True,
            **vllm_params,
        },
    )

    return [choice.token_ids for choice in response.choices]  # type: ignore[attr-defined]


def token_logprobs(prompt_ids: list[int], continuation: list[int]) -> np.ndarray:
    """Logprob каждого токена продолжения."""
    response = client.completions.create(
        model=MODEL,
        prompt=prompt_ids + continuation,
        max_tokens=1,
        extra_body={"prompt_logprobs": 0},
    )
    entries = response.choices[0].prompt_logprobs[len(prompt_ids):]  # type: ignore[attr-defined]
    logprobs = [entry[str(token)]["logprob"] for token, entry in zip(continuation, entries)]

    return np.array(logprobs)


prompt_ids = tokenizer("Кот спит на")["input_ids"]
greedy_ids = generate_ids(prompt_ids, 4)
beams = generate_ids(
    prompt_ids,
    4,
    n=4,
    use_beam_search=True,
)

paths = {"жадный": greedy_ids[0]}
for i, path in enumerate(beams, 1):
    paths[f"beam {i}"] = path
logprobs = {name: token_logprobs(prompt_ids, path) for name, path in paths.items()}

table = pd.DataFrame(
    {
        "продолжение": [tokenizer.decode(path) for path in paths.values()],
        "сумма logprobs": [lp.sum() for lp in logprobs.values()],
        "вероятность": [np.exp(lp.sum()) for lp in logprobs.values()],
    },
    index=list(paths),
)
table.round(4)

Вероятности по токенам для жадного пути и самого вероятного из найденных:

In [ ]:
best = max(logprobs, key=lambda name: logprobs[name].sum())

for name in dict.fromkeys(["жадный", best]):
    probabilities = np.exp(logprobs[name])
    tokens = [f"{piece(t)} {p:.2f}" for t, p in zip(paths[name], probabilities)]
    print(f"{name:7}", "   ".join(tokens))

Жадный выбор смотрит только на текущий шаг -- менее вероятный токен в начале может привести к более вероятному продолжению в итоге.

## Сэмплирование

Вместо самого вероятного токена можно выбирать случайный: каждый токен выбирается с той вероятностью, которую предсказала модель. Если `temperature` не задана, vLLM задаст её равной 1.

Параметр `n` просит несколько ответов на один запрос. Запросим несколько ответов длиной в один токен и сравним частоты токенов с вероятностями модели.

In [ ]:
def sample_first_tokens(
    prompt: str,
    n: int = 300,
    extra_body: dict[str, Any] | None = None,
    **params: Any,
) -> np.ndarray:
    """Доли первого токена в n ответах, по числу на каждый токен словаря; extra_body -- параметры vLLM."""
    response = client.completions.create(
        model=MODEL,
        prompt=prompt,
        max_tokens=1,
        n=n,
        extra_body={
            "return_token_ids": True,
            **(extra_body or {}),
        },
        **params,
    )
    first_tokens = [choice.token_ids[0] for choice in response.choices]

    return np.bincount(first_tokens, minlength=VOCAB) / n


lp = next_token_logprobs("Кот спит на")
sampled = sample_first_tokens("Кот спит на", temperature=1)
plot_distributions(
    {
        "модель": np.exp(lp),
        "доля в ответах сервера": sampled,
    },
    title='После "Кот спит на", T = 1',
)

Маловероятные токены по отдельности выпадают редко, но вместе набирают заметную долю -- см. столбец "остальные". В длинном тексте вероятность того что они будут выбраны повышается и модель может потерять корректную нить рассуждения. `logprobs=0` возвращает logprob выбранного токена

In [ ]:
completion = client.completions.create(
    model=MODEL,
    prompt="Кот спит на диване.",
    max_tokens=40,
    n=3,
    temperature=1,
    logprobs=0,
    extra_body={"return_token_ids": True},
)

for choice in completion.choices:
    assert choice.logprobs and choice.logprobs.token_logprobs
    parts = []
    for token, logprob in zip(choice.token_ids, choice.logprobs.token_logprobs):  # type: ignore[attr-defined]
        text = cast(str, tokenizer.decode([token]))
        if logprob < np.log(0.01):
            text = f"[{text}]"
        parts.append(text)

    print("".join(parts), end="\n\n")

## Температура

Перед тем как выбрать токен логиты делят на температуру $T$: $p = \mathrm{softmax}(z / T)$. Logprobs отличаются от логитов на константу, поэтому формула применима и к ним. При $T \rightarrow 0$ почти вся вероятность аккумулируется на элементе с максимальным логитом, при $T \rightarrow \infty$ -- размазывается более равномерно между всеми токенами.

In [ ]:
def softmax(z: np.ndarray) -> np.ndarray:
    e = np.exp(z - z.max())

    return e / e.sum()


def with_temperature(logprobs: np.ndarray, T: float) -> np.ndarray:
    return softmax(logprobs / T)


distributions = {f"T = {T}": with_temperature(lp, T) for T in [0.5, 1, 2]}
plot_distributions(distributions, title='После "Кот спит на"')

In [ ]:
def ask(
    prompt: str,
    system: str | None = None,
    extra_body: dict[str, Any] | None = None,
    **params: Any,
) -> list[Choice]:
    """Варианты ответа (choices) на одно сообщение пользователя; режим рассуждений выключен."""
    messages: list[ChatCompletionMessageParam] = []
    if system:
        messages.append({
            "role": "system",
            "content": system,
        })
    messages.append({
        "role": "user",
        "content": prompt,
    })

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        extra_body={
            **NO_THINK,
            **(extra_body or {}),
        },
        **params,
    )

    return response.choices


def coffee(**params: Any) -> list[str | None]:
    """Названия кофейни, которые придумала модель; params -- параметры генерации."""
    choices = ask(
        "Придумай название для кофейни. Только название.",
        max_tokens=20,
        **params,
    )

    return [choice.message.content for choice in choices]


names = {}
for T in [0, 0.7, 1, 1.5, 2]:
    names[f"T = {T}"] = [coffee(temperature=T)[0] for _ in range(3)]

pd.DataFrame(names).T

Жадный выбор уместен, когда ответ -- один токен. (Например, LLM-реранкер получает запрос и документ и отвечает "yes" или "no")

In [ ]:
QUERY = "Как вернуть наушники, если левый не работает?"
documents = [
    "Неисправный товар можно вернуть в течение 14 дней: оформите заявку в личном кабинете и приложите фото.",
    "Наушники с шумоподавлением подходят для перелётов и работы в открытом офисе.",
    "Доставка курьером занимает от одного до трёх дней.",
]


def relevance(document: str) -> dict[str, object]:
    """Ответ модели одним токеном и вероятности "yes" и "no" на его месте."""
    choices = ask(
        f"Query: {QUERY}\nDocument: {document}",
        system="Judge whether the document answers the query. Answer only yes or no.",
        max_tokens=1,
        temperature=0,
        logprobs=True,
        top_logprobs=20,
    )
    choice = choices[0]
    assert choice.logprobs and choice.logprobs.content

    p: defaultdict[str, float] = defaultdict(float)
    for top in choice.logprobs.content[0].top_logprobs:
        token = top.token.strip().lower()
        p[token] += np.exp(top.logprob)

    return {
        "ответ": choice.message.content,
        "p(yes)": p["yes"],
        "p(no)": p["no"],
        "оценка": p["yes"] / (p["yes"] + p["no"]),
    }


records = [relevance(d) for d in documents]
index = [d[:50] + "…" for d in documents]
table = pd.DataFrame(records, index=index)

table.sort_values("оценка", ascending=False).round(4)

## Top-k и top-p

Top-k оставляет k самых вероятных токенов.
Top-p оставляет некоторое число таких самых вероятных токенов, чтобы их сумма оказалась больше либо равна p (но если убрать хоть один токен, то была меньше).

In [ ]:
def keep_only(p: np.ndarray, ids: np.ndarray) -> np.ndarray:
    """Распределение только по токенам ids"""
    q = np.zeros_like(p)
    q[ids] = p[ids]

    return q / q.sum()


def top_k_filter(p: np.ndarray, k: int) -> np.ndarray:
    top = np.argsort(-p)[:k]

    return keep_only(p, top)


def top_p_filter(p: np.ndarray, top_p: float) -> np.ndarray:
    order = np.argsort(-p)
    cutoff = np.searchsorted(np.cumsum(p[order]), top_p) + 1

    return keep_only(p, order[:cutoff])


def describe(prompt: str) -> dict[str, object]:
    """Сколько токенов оставляет top-p и сколько вероятности отсекает top-k."""
    p = np.exp(next_token_logprobs(prompt))

    row: dict[str, object] = {"самый вероятный": f"{piece(p.argmax())} {p.max():.3f}"}
    for top_p in (0.9, 0.95):
        row[f"токенов в top_p {top_p}"] = np.count_nonzero(top_p_filter(p, top_p))
    for k in (20, 40):
        row[f"вероятность вне top_k {k}"] = 1 - np.sort(p)[-k:].sum()

    return row


prefixes = [
    "Москва -- столица России, а Париж -- столица",
    "Кот спит",
    "Кот спит на",
    "Однажды",
]
records = [describe(prompt) for prompt in prefixes]

pd.DataFrame(records, index=prefixes).round(3)

Число кандидатов у top-k фиксировано, а у top-p зависит от распределения: например, один токен может уже преодолеть порог.

В vLLM параметры применяются в следующем порядке: температура, top-k, top-p.

В некоторых других движках температуру применяют после top-p и top-k.

`top_k` -- параметр vLLM

In [ ]:
def vllm_order(logprobs: np.ndarray, T: float, top_k: int, top_p: float) -> np.ndarray:
    p = with_temperature(logprobs, T)
    p = top_k_filter(p, top_k)

    return top_p_filter(p, top_p)


def filters_first(logprobs: np.ndarray, T: float, top_k: int, top_p: float) -> np.ndarray:
    p = top_k_filter(np.exp(logprobs), top_k)
    kept = top_p_filter(p, top_p) > 0
    masked = np.where(kept, logprobs, -np.inf)

    return with_temperature(masked, T)


T = 2
K = 4
P = 0.8

server = sample_first_tokens(
    "Кот спит на",
    temperature=T,
    top_p=P,
    extra_body={"top_k": K},
)
plot_distributions(
    {
        "T -> top-k -> top-p": vllm_order(lp, T, K, P),
        "top-k -> top-p -> T": filters_first(lp, T, K, P),
        "доля в ответах сервера": server,
    },
    title=f'После "Кот спит на": temperature {T}, top_k {K}, top_p {P}',
    k=K,
)

Для обычных задач без рассуждений разработчики Qwen3.5 рекомендуют `temperature=0.7`, `top_p=0.8` и `top_k=20`.

## Seed и несколько ответов

Seed задаёт начальное состояние генератора случайных чисел запроса. С тем же seed при тех же вероятностях выбираются те же токены, с другим seed ответ может измениться. Запрос с `n` > 1 vLLM разбивает на n дочерних, и при заданном seed дочерний запрос с номером i получает seed + i.

In [ ]:
names = {
    "без seed": [coffee(temperature=1)[0] for _ in range(3)],
    "seed 42 три раза": [coffee(temperature=1, seed=42)[0] for _ in range(3)],
    "seed 42, 43, 44": [coffee(temperature=1, seed=seed)[0] for seed in (42, 43, 44)],
    "n = 3, seed 42": coffee(temperature=1, n=3, seed=42),
}

pd.DataFrame(names).T

`n` задаёт число ответов, но не выбирает лучший: в отличие от beam search, вероятности ответов не сравниваются и они могут повторяться.

Seed фиксирует случайные числа, но не вычисления. Сервер считает очередной токен сразу для всех текущих запросов, и от их числа зависит порядок суммирования; из-за округления логиты немного меняются, и при близких вероятностях может выбраться другой токен.

## Завершение генерации

Модель выдаёт токены $y_1, y_2, \dots$ и останавливается на первом шаге $t$, где выполнено одно из условий. Причину сервер возвращает в `finish_reason` и в поле vLLM `stop_reason`.

| условие | `finish_reason` | `stop_reason` |
|---|---|---|
| $y_t = \mathrm{EOS}$, токен конца сообщения | `stop` | `None` |
| текст $y_1 \dots y_t$ содержит строку $s$ из `stop` | `stop` | $s$ |
| $t$ = `max_tokens` | `length` | `None` |

In [ ]:
settings: list[dict[str, Any]] = [
    {"max_tokens": 50},
    {
        "max_tokens": 50,
        "stop": [","],
    },
    {"max_tokens": 2},
]

for params in settings:
    choices = ask(
        "Назови три цвета радуги через запятую.",
        temperature=0,
        **params,
    )
    answer = choices[0]
    stop_reason = answer.stop_reason  # type: ignore[attr-defined]

    print(params)
    print(f"    finish_reason={answer.finish_reason}, stop_reason={stop_reason!r}: {answer.message.content!r}")

Stop-строка в ответ не входит и не обязана совпадать с одним токеном. `max_tokens` ограничивает каждый ответ отдельно, в том числе при `n` > 1.

## Messages, роли, история диалога

Запрос к чат-модели -- список сообщений messages. У каждого сообщения есть роль:
- `system` -- для начальных инструкций;
- `user` -- ответов пользователя;
- `assistant` -- прошлых ответов модели

Прошлые ответы `assistant` нужно передавать в каждом запросе, они составляют единый контекст. Модель не помнит что с ней было ранее и что она отвечала.

In [ ]:
system = {
    "role": "system",
    "content": "Помогай с возвратами. Определяй следующий шаг с учётом истории обращения. Отвечай кратко.",
}
first = {
    "role": "user",
    "content": "Хочу вернуть наушники из заказа №7421.",
}
approved = {
    "role": "assistant",
    "content": "Возврат по заказу №7421 одобрен. Деньги поступят на карту завтра.",
}
followup = {
    "role": "user",
    "content": "Когда поступят деньги?",
}

dialog = [system, first, approved, followup]
without_answer = [system, first, followup]
without_history = [system, followup]

texts = chat_parallel((dialog, without_answer, without_history), max_tokens=80)
for text in texts:
    print(text, end="\n\n")

## Chat template

Модель получает не список сообщений, а одну последовательность токенов. Текст из списка собирает chat template -- шаблон Jinja из репозитория модели. Применим шаблон через `transformers` и напечатаем рядом текст, который собрал сервер: его возвращает параметр vLLM `return_prompt_text`.

In [ ]:
first_turn = [system, first]
text = cast(str, tokenizer.apply_chat_template(
    first_turn,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
))

response = client.chat.completions.create(
    model=MODEL,
    messages=cast(list[ChatCompletionMessageParam], first_turn),
    max_tokens=1,
    extra_body={
        "return_prompt_text": True,
        **NO_THINK,
    },
)

show({
    "transformers": text,
    "сервер": response.prompt_text,  # type: ignore[attr-defined]
})

Пустой блок `<think></think>` в конце добавил шаблон, так как режим рассуждений выключен. Служебные маркеры `<|im_start|>` и `<|im_end|>` -- отдельные токены:

In [ ]:
ids = tokenizer(text)["input_ids"]
print(len(ids), "токенов")

pieces = [cast(str, tokenizer.decode([i])).replace("\n", "\\n") for i in ids]
print(" | ".join(pieces))

Ответ модели -- продолжение текста, собранного шаблоном. Передадим этот текст в `/v1/completions` и напечатаем его вместе с продолжением, а затем ответ `/v1/chat/completions` на те же messages.

Модель заканчивает ответ токеном `<|im_end|>` -- тем же, что стоит в шаблоне в конце каждого сообщения. Это токен конца: на нём генерация останавливается.

In [ ]:
completion = client.completions.create(
    model=MODEL,
    prompt=text,
    max_tokens=100,
    temperature=0,
    extra_body={"return_token_ids": True},
)
choice = completion.choices[0]
continuation = cast(str, tokenizer.decode(choice.token_ids))  # type: ignore[attr-defined]
reply = chat(first_turn, max_tokens=100).choices[0].message.content

show({
    "/v1/completions": text + continuation,
    "/v1/chat/completions": reply,
})
print("finish_reason:", choice.finish_reason)

## System prompt

Системный промпт -- сообщение с ролью `system` перед репликами пользователя. В нём разработчик приложения задаёт модели задачу, поведение и формат ответа. Отправим одно и то же сообщение пользователя без системного промпта и с тремя разными.

In [ ]:
user_message = {
    "role": "user",
    "content": "Хочу вернуть наушники, левый не работает.",
}
systems = {
    "helpful assistant": "You are a helpful assistant.",
    "сотрудник поддержки": (
        "Ты сотрудник поддержки. Помоги пользователю оформить возврат. Уточняй недостающие сведения."
    ),
    "классификатор": (
        "Классифицируй обращения: доставка, возврат, оплата. Ответ должен содержать только название категории."
    ),
}

variants = {"без system": [user_message]}
for name, content in systems.items():
    variants[name] = [
        {
            "role": "system",
            "content": content,
        },
        user_message,
    ]

texts = chat_parallel(variants.values(), max_tokens=120)
show(dict(zip(variants, texts)))

Без системного промпта шаблон Qwen3.5 ничего не добавляет от себя, как, например, сделал бы шаблон Qwen2.5, который подставляет системное сообщение по умолчанию.

Порой тут кроется тонкость того, что при использовании модели и шаблонов надо четко понимать с какими шаблонами модель училась, иначе можно получать плохое качество генерации просто потому что инструмент используется не совсем правильно

In [ ]:
templates = {
    "Qwen2.5": AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct"),
    "Qwen3.5": tokenizer,
}

for name, model_tokenizer in templates.items():
    prompt = model_tokenizer.apply_chat_template(
        [user_message],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    print(f"=== {name}\n{prompt}")

## Инструкция, правила и примеры

Классификатор обращений -- часть сервиса: его ответ обрабатывает обычная программа.

In [ ]:
lines = Path("../data/support_cases.jsonl").read_text(encoding="utf-8").splitlines()
cases = [json.loads(line) for line in lines]

pd.DataFrame(cases).set_index("id")

Варианты запроса:

- инструкция: действие, допустимые категории, запасное "другое" и формат ответа;
- zero-shot: инструкция и правила маршрутизации;
- one-shot и few-shot: правила и один или три примера;
- few-shot парами: те же три примера отдельными сообщениями `user` и `assistant`.

Системная инструкция few-shot:

In [ ]:
INSTRUCTION = (
    "Определи категорию обращения по правилам маршрутизации.\n\n"
    "Выбери «доставка», «возврат» или «оплата». Если ни одна категория не подходит или данных для выбора "
    "недостаточно, выбери «другое».\n\n"
    "Ответ должен содержать только название категории в кавычках, без пояснений."
)
RULES = (
    "Правила маршрутизации\n"
    "«доставка»: получение и доставка заказа.\n"
    "«возврат»: возврат товара и получение денег за возвращённый товар.\n"
    "«оплата»: оплата покупки и списание денег."
)
EXAMPLES = [
    ("Курьер не приехал.", "«доставка»"),
    ("За покупку списали деньги дважды.", "«оплата»"),
    ("После возврата куртки жду зачисления.", "«возврат»"),
]


def with_examples(k: int) -> str:
    prompt = INSTRUCTION + "\n\n" + RULES
    if not k:
        return prompt

    shots = [f"Обращение: {text}\nКатегория: {label}" for text, label in EXAMPLES[:k]]

    return prompt + "\n\nПримеры\n" + "\n\n".join(shots)


def system_prompt(content: str) -> Callable[[str], list[dict[str, str]]]:
    def build(text: str) -> list[dict[str, str]]:
        return [
            {
                "role": "system",
                "content": content,
            },
            {
                "role": "user",
                "content": text,
            },
        ]

    return build


def examples_as_pairs(text: str) -> list[dict[str, str]]:
    messages = [
        {
            "role": "system",
            "content": with_examples(0),
        },
    ]
    for example, label in EXAMPLES:
        messages.append({
            "role": "user",
            "content": example,
        })
        messages.append({
            "role": "assistant",
            "content": label,
        })
    messages.append({
        "role": "user",
        "content": text,
    })

    return messages


VARIANTS = {
    "инструкция": system_prompt(INSTRUCTION),
    "zero-shot": system_prompt(with_examples(0)),
    "one-shot": system_prompt(with_examples(1)),
    "few-shot": system_prompt(with_examples(3)),
    "few-shot парами": examples_as_pairs,
}

print(with_examples(3))

Каждый вариант проверим двумя способами:

- на корректность категории -- категория из ответа, допустимы любые кавычки, совпадает с разметкой;
- формат -- ответ ровно `«категория»`

In [ ]:
LABELS = ["доставка", "возврат", "оплата", "другое"]


def label_of(reply: str) -> str | None:
    """Категория, если ответ -- одно название в любых кавычках."""
    match = re.fullmatch(r"""\s*["«“']?\s*(\w+)\s*["»”']?\s*\.?\s*""", reply)
    if not match:
        return None

    label = match.group(1).lower()

    return label if label in LABELS else None


def strict_format(reply: str) -> bool:
    """Формат как в правилах и примерах: «категория» и ничего больше."""
    return re.fullmatch(r"«(доставка|возврат|оплата|другое)»", reply.strip()) is not None


def classify_cases(build: Callable[[str], list[dict[str, str]]]) -> list[str]:
    """Ответы классификатора на все обращения; build собирает messages для одного обращения."""
    dialogs = [build(case["input"]) for case in cases]

    return chat_parallel(dialogs, max_tokens=15)


replies = {name: classify_cases(build) for name, build in VARIANTS.items()}

expected = [case["expected"].strip("«»") for case in cases]
summary = {}
for name in VARIANTS:
    summary[name] = {
        f"категория верна, из {len(cases)}": sum(label_of(r) == e for r, e in zip(replies[name], expected)),
        f"формат «…», из {len(cases)}": sum(map(strict_format, replies[name])),
    }

pd.DataFrame(summary).T

Ответы по обращениям:

In [ ]:
columns = {
    "обращение": [case["input"] for case in cases],
    "feature": [case["feature"] for case in cases],
    "ожидается": [case["expected"] for case in cases],
    **replies,
}

pd.DataFrame(columns)

## Постановка задачи

Следующая задача бота -- рассчитать сумму возврата. В системном промпте постановка задача и бизнес-правила, в user -- данные заказа. 

In [ ]:
TASK = 'Рассчитай сумму возврата за возвращаемые товары и укажи её в рублях. Последней строкой напиши "Итого: N руб.".'
REFUND_RULES = (
    "По правилам скидка распределяется пропорционально стоимости товаров, доставка не является товаром, "
    "деньги за доставку не возвращаются."
)
order = {
    "role": "user",
    "content": (
        "В заказе наушники за 4000 рублей и чехол за 1000 рублей. Общая скидка на товары -- "
        "1000 рублей. Доставка -- 300 рублей. Возвращаются только наушники."
    ),
}
EXPECTED = 4000 - 1000 * 4000 // 5000  # на наушники приходится доля скидки, пропорциональная их цене


def total(reply: str) -> int | None:
    """Сумма из строки "Итого: N руб.", если модель её написала."""
    found = re.findall(r"Итого[^\d\n]*(\d[\d   ]*)", reply)
    if not found:
        return None

    digits = re.sub(r"\D", "", found[-1])

    return int(digits)


variants = {
    # вместо правил компании -- просьба быть точным
    'без правил, "будь точным"': [
        {
            "role": "system",
            "content": TASK + "\n\nБудь точным.",
        },
        order,
    ],
    "с правилами": [
        {
            "role": "system",
            "content": TASK + "\n\n" + REFUND_RULES,
        },
        order,
    ],
}
refund_texts = chat_parallel(variants.values(), max_tokens=800)
show(dict(zip(variants, refund_texts)))

# итог модель пишет последней строкой "Итого: N руб.", сравниваем его с суммой по правилам компании
pd.DataFrame(
    {
        "итог модели": [total(reply) for reply in refund_texts],
        "по правилам компании": EXPECTED,
    },
    index=list(variants),
)

## Chain-of-Thought

Задача -- рассчитать сумму возврата по правилам компании. Набор из восьми заказов с известными ответами: первый -- заказ с наушниками, остальные сгенерированы.

In [ ]:
RULES = (
    "По правилам скидка распределяется пропорционально стоимости товаров, доставка не является товаром, "
    "деньги за доставку не возвращаются."
)
ORDER = (
    "Наушники стоят 4000 руб., чехол 1000 руб. Скидка на товары 1000 руб., доставка 300 руб. "
    "Возвращаю только наушники."
)
ITEMS = ["наушники", "чехол", "зарядное устройство", "кабель", "колонка", "часы", "клавиатура", "мышь", "подставка"]


def make_task(rng: random.Random) -> dict[str, Any]:
    """Случайный заказ с целой суммой возврата: два или три товара, скидка, доставка, один или два возвращаемых."""
    while True:
        k = rng.choice([2, 3])
        names = rng.sample(ITEMS, k)
        prices = [rng.randrange(5, 80) * 100 for _ in range(k)]
        discount = rng.randrange(2, 30) * 100
        count = 1 if k == 2 else rng.choice([1, 2])
        returned = sorted(rng.sample(range(k), count))
        returned_sum = sum(prices[i] for i in returned)
        if discount < 0.6 * sum(prices) and discount * returned_sum % sum(prices) == 0:
            break

    goods = ", ".join(f"{name} за {price} рублей" for name, price in zip(names, prices))
    delivery = rng.randrange(2, 8) * 100
    returned_names = ", ".join(names[i] for i in returned)
    text = (
        f"В заказе {goods}. Общая скидка на товары -- {discount} рублей. Доставка -- {delivery} "
        f"рублей. Возвращаемые товары: {returned_names}."
    )
    answer = returned_sum - discount * returned_sum // sum(prices)

    return {
        "text": text,
        "answer": answer,
    }


rng = random.Random(7)
tasks = [
    {
        "text": ORDER,
        "answer": 4000 - 1000 * 4000 // 5000,
    },
]
tasks += [make_task(rng) for _ in range(7)]

pd.DataFrame(tasks)

Три постановки:

- только ответ: итоговая сумма без вычислений;
- zero-shot CoT: инструкция решать по шагам;
- few-shot CoT: та же инструкция и два решённых примера на других числах.

Во всех постановках итог -- последняя строка "Сумма возврата: … руб.", из неё берём ответ.

In [ ]:
FORMAT = 'Последней строкой напиши "Сумма возврата: <число> руб.".'
COT = "Реши задачу по шагам, затем укажи итоговую сумму возврата в рублях. " + FORMAT
COT_EXAMPLES = (
    "Примеры решения\n\n"
    "Товары: 3000 и 2000 руб. Скидка: 500 руб. Возврат первого товара.\n"
    "500 · 3000 / 5000 = 300; 3000 − 300 = 2700.\n"
    "Сумма возврата: 2700 руб.\n\n"
    "Товары: 1000 и 3000 руб. Скидка: 400 руб. Возврат первого товара.\n"
    "400 · 1000 / 4000 = 100; 1000 − 100 = 900.\n"
    "Сумма возврата: 900 руб."
)
INSTRUCTIONS = {
    "только ответ": (
        "Рассчитай сумму возврата за возвращаемые товары. Ответь только итоговой суммой, без вычислений, "
        'в формате "Сумма возврата: <число> руб.".'
    ),
    "zero-shot CoT": COT,
    "few-shot CoT": COT + "\n\n" + COT_EXAMPLES,
}
NUMBER = r"\d[\d   ]*(?:[.,]\d+)?"


def refund(instruction: str, order: str, rules: str | None = RULES) -> list[dict[str, str]]:
    """Запрос на расчёт: в system -- инструкция и правила компании, в user -- заказ."""
    system = instruction
    if rules:
        system += "\n\n" + rules

    return [
        {
            "role": "system",
            "content": system,
        },
        {
            "role": "user",
            "content": order,
        },
    ]


def amount(text: str | None) -> int | float | None:
    """Сумма возврата: первое число перед "руб" после последних слов "сумма возврата" в строке."""
    pattern = r"сумма возврата[^\n]*?(" + NUMBER + r")\W*руб"
    found = re.findall(pattern, text or "", re.IGNORECASE)
    if not found:
        return None

    digits = re.sub(r"\s", "", found[-1]).replace(",", ".")
    value = float(digits)

    return int(value) if value.is_integer() else value


def generate(messages: Iterable[Message], **params: Any) -> tuple[list[str | None], int]:
    """Тексты ответов (n штук) и число сгенерированных токенов."""
    response = chat(messages, **params)
    texts = [choice.message.content for choice in response.choices]
    assert response.usage is not None

    return texts, response.usage.completion_tokens


def solve(task: dict[str, Any], instruction: str = COT) -> dict[str, Any]:
    """Одно решение при жадном выборе; ответ -- сумма из последней строки."""
    messages = refund(instruction, task["text"])
    texts, tokens = generate(messages)

    return {
        "решение": texts[0],
        "сумма": amount(texts[0]),
        "запросов": 1,
        "токенов": tokens,
    }


def score(results: list[dict[str, Any]]) -> dict[str, object]:
    """Верные суммы на наборе и токенов на задачу."""
    right = sum(r["сумма"] == task["answer"] for r, task in zip(results, tasks))
    tokens = [r["токенов"] for r in results]

    return {
        "верно": f"{right}/{len(results)}",
        "токенов на задачу": round(np.mean(tokens)),
    }


greedy = {}
for name, instruction in INSTRUCTIONS.items():
    greedy[name] = parallel(lambda task: solve(task, instruction), tasks)

scores = {name: score(results) for name, results in greedy.items()}
pd.DataFrame(scores).T

Ответы по задачам:

In [ ]:
columns = {
    "задача": [task["text"] for task in tasks],
    "верно": [task["answer"] for task in tasks],
}
for name, runs in greedy.items():
    columns[name] = [r["сумма"] for r in runs]

pd.DataFrame(columns)

Решения заказа с наушниками по инструкции и по примерам:

In [ ]:
solutions = {name: greedy[name][0]["решение"] for name in ["zero-shot CoT", "few-shot CoT"]}
show(solutions)

## Self-Consistency

Self-Consistency выбирает самый вероятный итог, а не самую вероятную цепочку рассуждений. Один итог $a$ на запрос $x$ получается разными цепочками $r$, и их вероятности складываются: $p(a \mid x) = \sum_r p(r, a \mid x)$.

Перебрать все цепочки нельзя, поэтому $p(a \mid x)$ оценивают долей итога $a$ среди $n$ решений со случайным выбором токенов. При жадном выборе все решения одинаковы.

Это вероятность итога при выбранных параметрах генерации, а не его правильность: ошибка, общая для многих решений, тоже может победить в голосовании.

Каждое решение -- отдельная генерация, поэтому затраты растут с $n$.

In [ ]:
SAMPLING = {
    "temperature": 0.7,
    "top_p": 0.8,
    "extra_body": {"top_k": 20},
}


def majority(amounts: list[int | float | None]) -> int | float | None:
    """Самая частая сумма; ответы без суммы не голосуют."""
    votes = Counter(a for a in amounts if a is not None)
    if not votes:
        return None

    return votes.most_common(1)[0][0]


def self_consistency(task: dict[str, Any], n: int = 10) -> dict[str, Any]:
    """n решений zero-shot CoT со случайным выбором токенов; ответ -- самая частая сумма."""
    messages = refund(COT, task["text"])
    texts, tokens = generate(
        messages,
        n=n,
        seed=0,
        **SAMPLING,
    )
    amounts = [amount(text) for text in texts]

    return {
        "решения": texts,
        "суммы": amounts,
        "сумма": majority(amounts),
        "запросов": 1,
        "токенов": tokens,
    }


voted = self_consistency(tasks[0])
print("суммы:", voted["суммы"])

votes = Counter(voted["суммы"]).most_common()
print("голоса:", dict(votes))
print(f"выбрано: {voted['сумма']}, верно: {tasks[0]['answer']}, токенов на все решения: {voted['токенов']}")

Решения с разными итогами, по одному на каждую сумму:

In [ ]:
by_amount = {f"сумма {a}": text for a, text in zip(voted["суммы"], voted["решения"])}
show(by_amount)

## Reasoning

Qwen3.5 обучена рассуждать перед ответом.

In [ ]:
# Рецепты рекоменуют
THINKING = {
    "temperature": 1.0,
    "top_p": 0.95,
    "seed": 0,
    "max_tokens": 6000,
}
COLUMNS = ["токенов рассуждения", "токенов всего", "finish_reason", "сумма", "секунд"]


def reason(
    messages: Iterable[Message],
    think: bool = True,
    budget: int | None = None,
    **params: Any,
) -> dict[str, Any]:
    """Ответ в режиме рассуждений или без него: рассуждение, ответ, сумма, токены и время."""
    extra_body = {"top_k": 20}
    if budget is not None:
        extra_body["thinking_token_budget"] = budget

    start = time.time()
    response = chat(
        messages,
        think=think,
        extra_body=extra_body,
        **(THINKING | params),
    )
    seconds = round(time.time() - start)
    choice = response.choices[0]
    usage = response.usage
    assert usage is not None and usage.completion_tokens_details is not None

    return {
        "рассуждение": choice.message.reasoning,  # type: ignore[attr-defined]
        "ответ": choice.message.content,
        "сумма": amount(choice.message.content),
        "finish_reason": choice.finish_reason,
        "токенов рассуждения": usage.completion_tokens_details.reasoning_tokens,
        "токенов всего": usage.completion_tokens,
        "секунд": seconds,
    }


lecture = refund(INSTRUCTIONS["только ответ"], ORDER)
outcomes = parallel(lambda think: reason(lecture, think), [False, True])
modes = dict(zip(["без рассуждения", "с рассуждением"], outcomes))

pd.DataFrame(modes).T[COLUMNS]

Рассуждение сервер возвращает в поле `reasoning`, ответ -- в `content`:

In [ ]:
thought = modes["с рассуждением"]
reasoning = thought["рассуждение"] or ""

show({
    "reasoning (начало)": reasoning[:1500],
    "content": thought["ответ"],
})

Передадим ответ вместе с рассуждением в истории и посмотрим текст, который получит модель:

In [ ]:
history = lecture + [
    {
        "role": "assistant",
        "content": thought["ответ"],
        "reasoning": thought["рассуждение"],
    },
    {
        "role": "user",
        "content": "А если вернуть и чехол?",
    },
]
prompt = rendered(history)

reasoning_start = thought["рассуждение"].strip()[:100]
print("рассуждение в тексте:", reasoning_start in prompt)

answer_start = prompt.index("<|im_start|>assistant")
print(prompt[answer_start:])

## Объём рассуждения

vLLM передаёт `reasoning_effort` в chat template. Текст промпта с уровнями effort и с `enable_thinking=False` в сравнении с текстом по умолчанию:

In [ ]:
default = rendered(lecture)

prompts = {}
for effort in ["low", "high", "none"]:
    prompts[f"reasoning_effort={effort}"] = rendered(lecture, reasoning_effort=effort)
prompts["enable_thinking=False"] = rendered(lecture, extra_body=NO_THINK)

pd.DataFrame({
    "как по умолчанию": {name: text == default for name, text in prompts.items()},
    "конец промпта": {name: repr(text[-30:]) for name, text in prompts.items()},
})

`reasoning_effort` -- указание модели: уровень передаётся вместе с запросом, а сколько рассуждать, решает модель, если её обучили различать уровни. В шаблоне Qwen3.5 уровней **нет**, есть только включение и выключение рассуждения.

`thinking_token_budget` -- лимит сервера: vLLM считает токены рассуждения и, когда их становится столько, сколько задано, ставит токен конца рассуждения и модель приступает к генерации ответа. Наша модель как раз принимает такой параметр

`max_tokens` ограничивает суммарно рассуждение и ответ.

In [ ]:
limits: dict[str, dict[str, Any]] = {
    "thinking_token_budget=256": {"budget": 256},
    "thinking_token_budget=64": {"budget": 64},
    "max_tokens=200": {"max_tokens": 200},
}
outcomes = parallel(lambda params: reason(lecture, **params), list(limits.values()))
modes |= dict(zip(limits, outcomes))

pd.DataFrame(modes).T[COLUMNS + ["ответ"]]

## Постановка задачи с рассуждением

Рассуждение не сообщает модели бизнес-правила. Та же постановка "только ответ" в режиме рассуждений без правил:

In [ ]:
without_rules = refund(INSTRUCTIONS["только ответ"], ORDER, None)
rules = {
    "с правилами": modes["с рассуждением"],
    "без правил": reason(without_rules),
}

pd.DataFrame(rules).T[COLUMNS]

Конец рассуждения и ответ без правил:

In [ ]:
no_rules = rules["без правил"]
reasoning = no_rules["рассуждение"] or ""

show({
    "reasoning (конец)": reasoning[-1500:],
    "content": no_rules["ответ"],
})

## Формат ответа

Программе, которая обрабатывает ответ бота, нужны отдельные поля: категория, сумма возврата и пересказ. Вместо выпаршивания их из текста можно попросить модель сразу вернуть JSON:

- попросить в инструкции JSON-объект;
- попросить в инструкции Markdown-блок с JSON;
- попросить JSON-объект и передать JSON Schema в параметре `response_format`.

Схему удобно задать Pydantic-моделью: по классу строится JSON Schema, и тем же классом программа проверяет ответ.

In [ ]:
class RefundResult(BaseModel):
    model_config = ConfigDict(extra="forbid")
    category: Literal["доставка", "возврат", "оплата", "другое"]
    refund_rub: int | None
    summary: str = Field(min_length=1, max_length=120)


schema = RefundResult.model_json_schema()
print(json.dumps(schema, ensure_ascii=False, indent=2))

Каждый ответ проверим по критериям:

- JSON -- `json.loads` не кидает исключений;
- схема -- JSON проходит валидацию `RefundResult`;
- категория и сумма совпадают с разметкой.

In [ ]:
# обращения с известной категорией, суммы для них нет, и заказы с известной суммой возврата
REFUNDS = {
    "Товары: 4000 и 1000 руб. Скидка: 1000 руб. Вернуть первый товар.": 3200,
    "Товары: 3000 и 2000 руб. Скидка: 500 руб. Вернуть первый товар.": 2700,
    "Товары: 1000 и 3000 руб. Скидка: 400 руб. Вернуть первый товар.": 900,
}

inputs = []
for case in cases:
    inputs.append({
        "text": case["input"],
        "category": case["expected"].strip("«»"),
        "refund_rub": None,
    })
for text, rub in REFUNDS.items():
    inputs.append({
        "text": text,
        "category": "возврат",
        "refund_rub": rub,
    })

TASK = (
    "Определи категорию обращения в поддержку магазина: «доставка», «возврат», «оплата» или «другое». Если ни одна "
    "категория не подходит или данных для выбора недостаточно, выбери «другое». Если в обращении есть данные для "
    "расчёта, рассчитай сумму возврата в рублях. Кратко перескажи обращение.\n\n"
    "Правила маршрутизации\n"
    "«доставка»: получение и доставка заказа.\n"
    "«возврат»: возврат товара и получение денег за возвращённый товар.\n"
    "«оплата»: оплата покупки и списание денег.\n\n"
    "По правилам скидка распределяется пропорционально стоимости товаров."
)
FIELDS = (
    "category должен содержать категорию, refund_rub -- сумму возврата целым числом или null, если данных "
    "для расчёта нет, summary -- краткий пересказ обращения."
)
SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "RefundResult",
        "schema": RefundResult.model_json_schema(),
        "strict": True,
    },
}
FORMATS: dict[str, dict[str, Any]] = {
    "JSON по инструкции": {"instruction": "Верни JSON-объект без пояснений. " + FIELDS},
    "Markdown-блок по инструкции": {"instruction": "Верни Markdown-блок для JSON. " + FIELDS},
    "JSON по схеме": {
        "instruction": "Верни JSON-объект без пояснений. " + FIELDS,
        "response_format": SCHEMA,
    },
}


def respond(item: dict[str, Any], instruction: str, **params: Any) -> str:
    messages = [
        {
            "role": "system",
            "content": TASK + "\n\n" + instruction,
        },
        {
            "role": "user",
            "content": item["text"],
        },
    ]
    response = chat(messages, **params)

    return response.choices[0].message.content or ""


def check(text: str, expected: dict[str, Any]) -> dict[str, bool]:
    """Что из ответа может использовать программа: JSON, схема, категория и сумма."""
    result = {
        "JSON": False,
        "схема": False,
        "категория": False,
        "сумма": False,
    }
    try:
        json.loads(text)
        result["JSON"] = True
        parsed = RefundResult.model_validate_json(text)
        result["схема"] = True
    except (json.JSONDecodeError, ValidationError):
        return result

    result["категория"] = parsed.category == expected["category"]
    result["сумма"] = parsed.refund_rub == expected["refund_rub"]

    return result


replies = {}
for name, variant in FORMATS.items():
    replies[name] = parallel(lambda item: respond(item, **variant), inputs)

counts = {}
for name, texts in replies.items():
    checks = [check(text, item) for text, item in zip(texts, inputs)]
    counts[name] = pd.DataFrame(checks).sum()

table = pd.DataFrame(counts).T
table.rename(columns=lambda column: f"{column}, из {len(inputs)}")

Схема ограничивает и длину полей: `summary` не длиннее 120 символов. В промпте об этом не сказано ни слова, но длинный пересказ после применения схемы обрывается на 120-м символе.

In [ ]:
# ответы на заказ с возвратом 3200 руб.
refund_replies = {name: texts[len(cases)] for name, texts in replies.items()}
show(refund_replies)

## Ограничение генерации

Схема ограничивает генерацию на каждом шаге: токены, с которыми ответ нарушит схему, получают нулевую вероятность -- чем-то похоже на top-k, но остаются не самые вероятные токены, а допустимые схемой, и их число k меняется от шага к шагу.

Поэтому схема сильнее инструкции: если в инструкции будет просьба генерировать ответ как Markdown-блок, а схема потребует JSON-объект, ответ начнётся с `{`, поскольку начало с backticks будет невалидным и исключится из генерации -- модель при этом даже не будет знать, что кто-то ограничивает её генерацию.

In [ ]:
# инструкция просит Markdown-блок; вероятности первого токена ответа без схемы и со схемой
messages = [
    {
        "role": "system",
        "content": TASK + "\n\n" + FORMATS["Markdown-блок по инструкции"]["instruction"],
    },
    {
        "role": "user",
        "content": inputs[len(cases)]["text"],
    },
]


def first_token(**params: Any) -> dict[str, float]:
    response = chat(
        messages,
        logprobs=True,
        top_logprobs=5,
        max_tokens=1,
        **params,
    )
    choice = response.choices[0]
    assert choice.logprobs and choice.logprobs.content
    first = choice.logprobs.content[0]

    return {repr(t.token): round(math.exp(t.logprob), 3) for t in first.top_logprobs}


table = pd.DataFrame({
    "без схемы": first_token(),
    "со схемой": first_token(response_format=SCHEMA),
})
table.fillna(0)

## Схема в клиенте

Метод `parse` клиента OpenAI принимает Pydantic-модель: строит из неё схему, передаёт серверу и применяет её к полученному ответу, возвращая python-объект.

In [ ]:
request: list[ChatCompletionMessageParam] = [
    {
        "role": "system",
        "content": TASK,
    },
    {
        "role": "user",
        "content": inputs[len(cases)]["text"],
    },
]
response = client.chat.completions.parse(
    model=MODEL,
    messages=request,
    response_format=RefundResult,
    temperature=0,
    extra_body=NO_THINK,
)
message = response.choices[0].message
assert message.parsed is not None

print(repr(message.parsed))
print("refund_rub:", message.parsed.refund_rub)

## Описание инструмента

Инструмент -- функция, вызов которой модель может запросить. Модель получает описание инструмента: имя, назначение и JSON Schema аргументов. Выполняет функцию сторонняя программа/сервер/приложение.

С `strict: true` сервер ограничивает генерацию аргументов схемой, как `response_format` ограничивает ответ.

In [ ]:
# данные возвратов вместо API магазина
RETURNS = {
    "7421": {
        "status": "payment_sent",
        "refund_rub": 3200,
        "business_days_max": 3,
    },
    "5310": {
        "status": "payment_sent",
        "refund_rub": 1450,
        "business_days_max": 5,
    },
}


def get_return_status(order_id: str) -> dict[str, object]:
    return RETURNS.get(order_id, {"error": "заказ не найден"})


GET_RETURN_STATUS = {
    "type": "function",
    "function": {
        "name": "get_return_status",
        "description": "Статус, сумма и срок возврата.",
        "parameters": {
            "type": "object",
            "properties": {"order_id": {"type": "string"}},
            "required": ["order_id"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}
QUESTION = [
    {
        "role": "user",
        "content": "Наушники уже приняли. Когда и сколько денег мне вернут по заказу №7421?",
    },
]

# модель видит инструмент как текст: шаблон добавил в системное сообщение его описание и формат вызова
prompt = rendered(QUESTION, tools=[GET_RETURN_STATUS], extra_body=NO_THINK)
print(prompt)

## Вызов инструмента

Модель генерирует вызов, как генерирует обычный текст, но в некотором формате, чтобы парсер смог найти этот вызов и положить в поле `tool_calls` (отдельно от `content`).

In [ ]:
# продолжение собранного промпта: вызов в том виде, в котором его пишет модель
completion = client.completions.create(
    model=MODEL,
    prompt=prompt,
    max_tokens=100,
    temperature=0,
)
raw = completion.choices[0].text

# тот же вопрос в чате: парсер нашёл вызов инструмента
response = chat(QUESTION, tools=[GET_RETURN_STATUS])
assistant = response.choices[0].message
calls = [call.model_dump() for call in assistant.tool_calls or []]

show({
    "генерация модели": raw,
    "content": assistant.content,
    "tool_calls": json.dumps(calls, ensure_ascii=False, indent=2),
})

При заданном параметре `tool_choice="auto"` модель сама решает, вызвать инструмент или ответить текстом, как обычно.

In [ ]:
questions = [
    "Когда вернут деньги по заказу 7421?",
    "Какие способы оплаты вы принимаете?",
    "Проверьте возвраты по заказам 7421 и 5310.",
    "Где мои деньги за возврат?",
]


def answer_with_tools(question: str) -> ChatCompletionMessage:
    """Ответ модели, которая сама решает, вызвать ли инструмент."""
    messages = [
        {
            "role": "user",
            "content": question,
        },
    ]
    response = chat(
        messages,
        tools=[GET_RETURN_STATUS],
        tool_choice="auto",
    )

    return response.choices[0].message


answers = parallel(answer_with_tools, questions)

tool_calls = []
for m in answers:
    tool_calls.append([f"{c.function.name}({c.function.arguments})" for c in m.tool_calls or [] if c.type == "function"])

pd.DataFrame({
    "обращение": questions,
    "вызовы": tool_calls,
    "текст": [m.content for m in answers],
})

## Цикл вызова

Стороннее приложение выполняет вызванный инструмент, а результат возвращает модели сообщением с ролью `tool`. Модель продолжает ответ с этими данными, "веря", что она вызвала инструмент и получила результат его работы. Цикл повторяется до тех пор, пока модель не ответит текстом.

In [ ]:
def run_tools(
    messages: Iterable[Message],
    tools: list[dict[str, Any]],
    functions: Mapping[str, Callable[..., object]],
    max_steps: int = 5,
) -> list[Message]:
    """История диалога после цикла вызовов инструментов."""
    messages = list(messages)
    for _ in range(max_steps):
        response = chat(messages, tools=tools)
        message = response.choices[0].message
        assistant = message.model_dump(
            include={"role", "content", "tool_calls"},
            exclude_none=True,
        )
        messages.append(assistant)
        if not message.tool_calls:
            break

        for call in message.tool_calls:
            assert call.type == "function"
            function = functions[call.function.name]
            arguments = json.loads(call.function.arguments)
            result = function(**arguments)
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result, ensure_ascii=False),
            })

    return messages


def transcript(messages: list[Message]) -> None:
    for m in messages:
        calls = [f"{c['function']['name']}({c['function']['arguments']})" for c in m.get("tool_calls", [])]
        content = m.get("content") or ""
        print(f"{m['role'].upper():10} {content}{' '.join(calls)}")


# заказа №9999 в данных магазина нет
unknown_order = [
    {
        "role": "user",
        "content": "Когда вернут деньги по заказу №9999?",
    },
]

for dialog in [QUESTION, unknown_order]:
    history = run_tools(
        dialog,
        [GET_RETURN_STATUS],
        {"get_return_status": get_return_status},
    )
    transcript(history)
    print()

## Оценщик

Ответ бота клиенту будем оценивать по двум критериям:

- достоверность -- все утверждения следуют из данных инструмента;
- полнота -- указаны и сумма, и срок.

Оценщик (LLM-as-a-judge) -- та же модель с инструкцией проверить критерии.

> По-хорошему так лучше не делать из-за предпочтения моделью собственной генерации. А ещё можно пробовать оценивать моделью побольше и умнее.

In [ ]:
class Criterion(BaseModel):
    quote: str = Field(max_length=200)
    reason: str = Field(max_length=300)
    verdict: Literal["да", "нет"]


class Judgement(BaseModel):
    faithful: Criterion
    complete: Criterion


JUDGE = (
    "Оцени ответ бота по вопросу пользователя и данным инструмента.\n\n"
    "faithful -- достоверность: все ли утверждения следуют из данных? Предельный срок не позволяет обещать "
    "конкретный день.\n"
    "complete -- полнота: указаны ли сумма и срок?\n\n"
    'Верни по каждому критерию цитату, причину и вердикт "да" или "нет".'
)


def judge(question: str, data: dict[str, object], answer: str) -> Judgement:
    data_json = json.dumps(data, ensure_ascii=False)
    request = f"Вопрос пользователя: {question}\n\nДанные инструмента: {data_json}\n\nОтвет бота: {answer}"
    messages: list[ChatCompletionMessageParam] = [
        {
            "role": "system",
            "content": JUDGE,
        },
        {
            "role": "user",
            "content": request,
        },
    ]
    completion = client.chat.completions.parse(
        model=MODEL,
        messages=messages,
        response_format=Judgement,
        temperature=0,
        max_tokens=1000,
        extra_body=NO_THINK,
    )
    judgement = completion.choices[0].message.parsed
    assert judgement is not None

    return judgement


LABELED = [  # вопрос, ответ бота, достоверность и полнота по оценке человека
    ("Когда и сколько денег мне вернут?", "Вам вернут 3200 рублей завтра.", "нет", "да"),
    ("Какую сумму ждать и когда?", "3200 рублей поступят в течение трёх рабочих дней.", "да", "да"),
    ("Когда придут деньги и сколько вернут?", "Деньги поступят в течение трёх рабочих дней.", "да", "нет"),
    ("Когда и сколько денег мне вернут?", "Возврат 3200 рублей придёт в течение 3 рабочих дней, обычно на следующий "
                                          "день.", "нет", "да"),
    ("Какую сумму ждать и когда?", "Вам вернут 4000 рублей в течение трёх рабочих дней.", "нет", "да"),
    ("Когда придут деньги и сколько вернут?", "Вам вернут 3200 рублей.", "да", "нет"),
    ("Успеете вернуть деньги до выходных?", "До выходных не успеем: срок возврата -- 3 рабочих дня.", "нет", "нет"),
    ("Деньги придут завтра?", "3200 рублей поступят не позже чем через 3 рабочих дня.", "да", "да"),
]
# проверка оценщика: его вердикты рядом с оценкой человека
labeled = parallel(lambda row: judge(row[0], RETURNS["7421"], row[1]), LABELED)

pd.DataFrame({
    "ответ бота": [row[1] for row in LABELED],
    "достоверность: человек": [row[2] for row in LABELED],
    "достоверность: оценщик": [j.faithful.verdict for j in labeled],
    "полнота: человек": [row[3] for row in LABELED],
    "полнота: оценщик": [j.complete.verdict for j in labeled],
})

## Сравнение версий промпта

Промпт из раза в раз надо оценивать на одних и тех же вопросах и данных, чтобы менялась только одна переменная за раз.

Поскольку ответы часто случайны (из-за сэмплирования), то каждую версию стоит запускать несколько раз: разброс оценок между повторами покажет влияние случайности.

In [ ]:
BOT = "Ты сотрудник поддержки магазина. Отвечай клиенту кратко по данным инструмента."
VERSIONS = {
    "текущая": BOT,
    "с правилом о сроке": (
        BOT + " Указывай срок из данных инструмента. Не заменяй предельный срок обещанием конкретного дня."
    ),
}
QUESTIONS = [
    "Когда и сколько денег мне вернут по заказу №{}?",
    "Деньги за заказ №{} придут завтра?",
    "Какую сумму ждать по заказу №{} и когда?",
    "Успеете вернуть деньги за заказ №{} до выходных?",
]

SCENARIOS = []
for order_id in RETURNS:
    for question in QUESTIONS:
        SCENARIOS.append((order_id, question.format(order_id)))


def bot_answers(system: str, order_id: str, question: str, n: int = 3) -> dict[str, Any]:
    """n ответов бота после вызова инструмента с данными заказа, токены и время."""
    call = {
        "id": "call_01",
        "type": "function",
        "function": {
            "name": "get_return_status",
            "arguments": json.dumps({"order_id": order_id}),
        },
    }
    messages: list[Message] = [
        {
            "role": "system",
            "content": system,
        },
        {
            "role": "user",
            "content": question,
        },
        {
            "role": "assistant",
            "content": "",
            "tool_calls": [call],
        },
        {
            "role": "tool",
            "tool_call_id": "call_01",
            "content": json.dumps(get_return_status(order_id), ensure_ascii=False),
        },
    ]

    start = time.time()
    response = chat(
        messages,
        tools=[GET_RETURN_STATUS],
        n=n,
        seed=0,
        temperature=0.7,
        top_p=0.8,
        extra_body={"top_k": 20},
        max_tokens=200,
    )
    seconds = time.time() - start
    assert response.usage is not None

    return {
        "answers": [choice.message.content or "" for choice in response.choices],
        "tokens": response.usage.completion_tokens,
        "seconds": seconds,
    }


results: dict[str, dict[str, Any]] = {}
for name, version in VERSIONS.items():
    generated = parallel(lambda scenario: bot_answers(version, *scenario), SCENARIOS)

    rows = []
    for (order_id, question), g in zip(SCENARIOS, generated):
        for repeat, answer in enumerate(g["answers"]):
            rows.append((order_id, question, repeat, answer))

    verdicts = parallel(lambda row: judge(row[1], RETURNS[row[0]], row[3]), rows)
    results[name] = {
        "rows": rows,
        "verdicts": verdicts,
        "generated": generated,
    }


def summarize(result: dict[str, Any]) -> dict[Hashable, object]:
    """Доли ответов, прошедших критерии, по всем повторам и разброс по повторам."""
    table = pd.DataFrame({
        "повтор": [row[2] for row in result["rows"]],
        "достоверно": [v.faithful.verdict == "да" for v in result["verdicts"]],
        "полно": [v.complete.verdict == "да" for v in result["verdicts"]],
    })
    table["оба критерия"] = table["достоверно"] & table["полно"]

    by_repeat = table.groupby("повтор")["оба критерия"].mean()
    shares = table[["достоверно", "полно", "оба критерия"]].mean().round(2).to_dict()
    tokens = sum(g["tokens"] for g in result["generated"])
    seconds = sum(g["seconds"] for g in result["generated"])

    return {
        **shares,
        "оба, по повторам": f"{by_repeat.min():.2f}–{by_repeat.max():.2f}",
        "токенов на ответ": round(tokens / len(result["rows"])),
        "секунд на запрос": round(seconds / len(SCENARIOS), 1),
    }


summaries = {name: summarize(result) for name, result in results.items()}
pd.DataFrame(summaries).T

Ответы, которые оценщик счёл недостоверными:

In [ ]:
unfaithful = []
for name, result in results.items():
    for row, v in zip(result["rows"], result["verdicts"]):
        if v.faithful.verdict == "нет":
            unfaithful.append({
                "версия": name,
                "вопрос": row[1],
                "ответ": row[3],
                "причина": v.faithful.reason,
            })

pd.DataFrame(unfaithful)